# JiT-S2-VMamba — in-context class tokens on the grid **diagonal**

8 in-context class tokens (`y_emb` + a learned slot, generated exactly as official JiT's) tied to the grid diagonal: token *e* sits just before cell (*e*, *e*) in canonical order and is **permuted together with the image** by SS2D's cross-scan, as Vim / Mamba-R's flip does for its registers: before its cell in paths 0/2, after it in the reversed paths 1/3. They enter at block 4 and stay through block 11 (`in_context_start: 4`, depth 12), then are stripped before the final layer. Scan `L = 64 + 8 = 72` (+12.5%) in blocks 4–11.

**Why this arm.** In a causal scan, what a token can reach depends on where it sits, so the three 8-token placements differ in what each token touches per block (real model, 3×3 DWConv included; the counts are reachability, not strength):

| placement | cells each token reads | cells each token writes to |
|---|---|---|
| prefix, K=8 | 0 for all | 64 for all |
| row | 0, 48, 60, 64, 64, 64, 64, 64 | 64, 64, 64, 64, 64, 60, 48, 28 |
| **diagonal (this arm)** | **64 for all** | **64 for all** |

One token per row **and** per column keeps the tokens evenly spaced (every 9 steps) in the row-major and the column-major scans. The diagonal itself is our choice: Mamba-R spaces its registers evenly in 1-D only, and its 2-D shape varies with the grid and register count.

**Reading the result.** Tokens, params and start block are the same in all three, so only the placement changes. These are class tokens, not label-free registers, so this tests placement, not registers as such.

| model (Tiny-ImageNet, 100 epochs) | params | FID-10K |
|---|---|---|
| prefix, K=8 (control) | 31,034,688 | ? |
| row (control) | 31,034,688 | ? |
| **this:** diagonal | 31,034,688 | ? |

The prefix K-sweep spans 0.31 FID, and row is already close to diagonal, so a **small or neutral gap to row is a plausible outcome**; a gap of a point or less needs a second seed to mean anything.

---

JiT-S2-VMamba on Tiny-ImageNet-200 (64px, patch 8, 200 classes) via the repo's `run_experiment.py` / `evaluate.py`. Comment out the download cell if you attach a dataset instead, and the eval cell if you only want to train.


## 1. Environment  *(Internet ON)*

In [ ]:
import os, sys

# Pick the stack by the image's Python. torch 2.5.1 has no matching torchvision
# for Python 3.13 (Kaggle's newer image), so 3.13 moves to torch 2.6.0; 3.12
# keeps torch 2.5.1 exactly as every earlier run. mamba-ssm 2.2.4 and
# causal-conv1d 1.5.0.post8 publish cp312 and cp313 wheels for both.
PY = f"cp{sys.version_info.major}{sys.version_info.minor}"
if sys.version_info >= (3, 13):
    TORCH, TVISION, TAUDIO, TTAG = "2.6.0", "0.21.0", "2.6.0", "torch2.6"
else:
    TORCH, TVISION, TAUDIO, TTAG = "2.5.1", "0.20.1", "2.5.1", "torch2.5"
print("python", sys.version.split()[0], "->", PY, "| torch", TORCH)

# 1) Pin torch
!pip install -q torch=={TORCH} torchvision=={TVISION} torchaudio=={TAUDIO} \
    --index-url https://download.pytorch.org/whl/cu124

# 2) Download wheels with explicit destination
CAUSAL = f"causal_conv1d-1.5.0.post8+cu12{TTAG}cxx11abiFALSE-{PY}-{PY}-linux_x86_64.whl"
MAMBA  = f"mamba_ssm-2.2.4+cu12{TTAG}cxx11abiFALSE-{PY}-{PY}-linux_x86_64.whl"

os.system(f"wget -q https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/{CAUSAL} -O /kaggle/working/{CAUSAL}")
os.system(f"wget -q https://github.com/state-spaces/mamba/releases/download/v2.2.4/{MAMBA} -O /kaggle/working/{MAMBA}")
for w in (CAUSAL, MAMBA):
    p = f"/kaggle/working/{w}"
    assert os.path.exists(p) and os.path.getsize(p) > 0, f"wheel download failed: {w}"

!pip install -q /kaggle/working/{CAUSAL}
!pip install -q /kaggle/working/{MAMBA}

# 3) Patch mamba-ssm
import glob
for path in glob.glob("/usr/local/lib/python*/dist-packages/mamba_ssm/utils/generation.py"):
    with open(path) as f: src = f.read()
    new = src.replace(
        "from transformers.generation import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput, TextStreamer",
        "from transformers.generation import GenerateDecoderOnlyOutput, TextStreamer",
    ).replace(
        "output_cls = GreedySearchDecoderOnlyOutput if top_k == 1 else SampleDecoderOnlyOutput",
        "output_cls = GenerateDecoderOnlyOutput",
    )
    if new != src:
        with open(path, "w") as f: f.write(new)
        print(f"\u2705 Patched {path}")

print(">>> RESTART RUNTIME NOW <<<")

## 2. Repo  *(clone + cd)*

In [ ]:
# Clone the repo. The diagonal layout (src/models/vmamba.py: diagonal_scan_index),
# the build_model wiring in BOTH run_experiment.py and evaluate.py, the config, the
# CPU test and the GPU check are all on main.
import os
REPO_DIR = "/kaggle/working/thesis_Choustoulakis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Rodamanthosch/thesis_Choustoulakis.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
# sanity: confirm the layout is in the model and wired into both entry points
!grep -q diagonal_scan_index src/models/vmamba.py \
    && grep -q 'in_context_layout=m.get' scripts/run_experiment.py && grep -q 'in_context_layout=m.get' scripts/evaluate.py \
    && echo "wiring OK (model + train + eval)" || echo "WIRING MISSING -- pull latest main"


## 3. Verify the arm  *(first session only; ~2-4 min)*

In [ ]:
# === Verify the diagonal arm (CPU proofs, ~1-2 min; first session only -- comment out on resume).
# Reference scan, no GPU: the gather with 0 tokens == cross_scan / cross_merge bit
# for bit; exact placement of every token and cell in all 4 paths against an
# independent derivation; gather -> scan -> inverse gather round trip; every token
# reads and writes every cell (row and prefix checked per cell too); state_dict
# identical to the prefix arm; scan length per block; guards.
# Check A compares against `git show HEAD:`, which on this clone already contains
# the diagonal code, so here it compares the code with itself; it passed against
# the previous commit before this arm was committed.
# PYTHONPATH=. is REQUIRED: without it "from src.models.vmamba import" fails
# and the tests' try/except misreports it as a missing mamba_ssm (known issue).
# If it FAILS, STOP -- do not train a model that is not the one reported.
!PYTHONPATH=. python tests/test_incontext_diagonal_cpu.py


In [ ]:
# === GPU check (~2-3 min; first session only). scripts/check_incontext_diagonal_cuda.py:
# 1) CUDA kernel vs mamba_ssm's selective_scan_ref through the gather path (forward
# and input gradient, fp32); 2) this config fwd+bwd under bf16 with the right scan
# length per block (64 x4, 72 x8); 3) it/s and peak memory at batch 128 vs prefix
# K=8 and row -- note them for the complexity table. If anything FAILS, STOP.
!PYTHONPATH=. python scripts/check_incontext_diagonal_cuda.py


## 4. Tiny-ImageNet  *(attached dataset, else download to /tmp)*

In [ ]:
# Tiny-ImageNet-200 -> /tmp (EPHEMERAL), ImageFolder layout for train/ and val/.
# Sources, in order: an attached Kaggle dataset containing tiny-imagenet-200/,
# then Stanford (https, http), then the SJTU mirror of the identical zip.
import os, zipfile, glob, shutil, subprocess, hashlib
DATA_DIR = "/tmp/tiny-imagenet-200"
ZIP      = "/tmp/tiny-imagenet-200.zip"
MD5      = "90528d7ca1a48142e341f4ef8d21d0de"
SOURCES  = ["https://cs231n.stanford.edu/tiny-imagenet-200.zip",
            "http://cs231n.stanford.edu/tiny-imagenet-200.zip",
            "https://scidata.sjtu.edu.cn/records/swpcp-01e87/files/tiny-imagenet-200.zip?download=1"]

def _md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    attached = [os.path.dirname(p) for p in
                glob.glob("/kaggle/input/**/tiny-imagenet-200/train", recursive=True)]
    if attached:
        print("copying attached dataset:", attached[0])
        shutil.copytree(attached[0], DATA_DIR)
    else:
        if os.path.exists(ZIP) and (not zipfile.is_zipfile(ZIP) or _md5(ZIP) != MD5):
            print(f"removing broken download ({os.path.getsize(ZIP)} bytes)")
            os.remove(ZIP)
        for src in SOURCES:
            if os.path.exists(ZIP):
                break
            print("downloading", src, "(~248MB)...")
            r = subprocess.run(["wget", "-q", "--tries=3", "--timeout=60", "-O", ZIP, src])
            ok = r.returncode == 0 and zipfile.is_zipfile(ZIP) and _md5(ZIP) == MD5
            if not ok:
                size = os.path.getsize(ZIP) if os.path.exists(ZIP) else 0
                print(f"  failed (wget exit {r.returncode}, {size} bytes)")
                if os.path.exists(ZIP):
                    os.remove(ZIP)
        assert os.path.exists(ZIP), ("Tiny-ImageNet download failed from every source -- "
                                     "attach a Kaggle tiny-imagenet dataset (Add Input) and rerun this cell")
        print("extracting...")
        with zipfile.ZipFile(ZIP) as z:
            z.extractall("/tmp")

# TRAIN: train/<cls>/images/*.JPEG -> train/<cls>/*.JPEG  (ImageFolder layout)
tr = os.path.join(DATA_DIR, "train")
for cls in os.listdir(tr):
    sub = os.path.join(tr, cls, "images")
    if os.path.isdir(sub):
        for f in glob.glob(os.path.join(sub, "*.JPEG")):
            shutil.move(f, os.path.join(tr, cls))
        shutil.rmtree(sub)
    for b in glob.glob(os.path.join(tr, cls, "*_boxes.txt")):
        os.remove(b)

# VAL: flat val/images + val_annotations.txt -> val/<cls>/*.JPEG  (evaluate.py reads val/)
val = os.path.join(DATA_DIR, "val")
ann = os.path.join(val, "val_annotations.txt")
if os.path.exists(ann):
    with open(ann) as f:
        for line in f:
            img, cls = line.split("\t")[:2]
            os.makedirs(os.path.join(val, cls), exist_ok=True)
            s = os.path.join(val, "images", img)
            if os.path.exists(s):
                shutil.move(s, os.path.join(val, cls, img))
    shutil.rmtree(os.path.join(val, "images"), ignore_errors=True)
    os.remove(ann)

n_tr, n_val = len(glob.glob(tr + "/*")), len(glob.glob(val + "/*"))
print("train classes:", n_tr, "| val classes:", n_val, "| DATA_DIR:", DATA_DIR)
assert n_tr == 200 and n_val == 200, "unexpected class count -- dataset incomplete"

## 5. Settings  *(edits the cloned config; finds the checkpoint to resume)*

In [ ]:
# ── Settings for THIS notebook ─────────────────────────────────────────
CONTENT    = "class"      # in_context_content -- the diagonal layout supports class only
LENGTH     = 8            # 8 tokens on cells (e, e); must divide grid_size 8 (4 -> every 2nd cell)
START      = 4            # tokens live in blocks START..11 (depth 12)
EPOCHS     = 100          # full target; resume across sessions until reached
SAVE_FREQ  = 1            # overwrite checkpoint-last.pt EVERY epoch (12h safety)

# Fallback if the download cell was commented out (attached-dataset workflow):
try:
    DATA_DIR
except NameError:
    DATA_DIR = "/kaggle/input/tiny-imagenet/tiny-imagenet-200"  # <-- your attached dataset
    print("DATA_DIR not set by a download cell; using:", DATA_DIR)

CONFIG  = "configs/tiny_imagenet/jit-s2-vmamba-incontext-diagonal-class.yaml"
RUN     = "exp_incontext_diagonal_class"
OUT_DIR = f"/kaggle/working/{RUN}"   # persists in the saved Version

# Sync the cloned config so BOTH training and evaluation read identical values
# (evaluate.py has no model CLI overrides -- it builds the model straight from
# config). The config sets no other arm, so the FFN, norm and adaLN stay at their
# defaults and the placement is the only variable vs row and prefix K=8.
import re
def set_cfg(path, kv):
    s = open(path).read()
    for k, v in kv.items():
        s, n = re.subn(rf"(?m)^(\s*{k}:).*$", rf"\1 {v}", s, count=1)
        assert n == 1, f"key {k!r} not found in {path}"
    open(path, "w").write(s)
set_cfg(CONFIG, {"data_dir": DATA_DIR, "d_state": 16, "in_context_layout": "diagonal",
                 "in_context_content": CONTENT, "in_context_len": LENGTH,
                 "in_context_start": START})

# ── Resume across sessions AND accounts (automatic) ────────────────────
# Looks for THIS model's newest checkpoint in: this session's OUT_DIR, any
# attached notebook output (.../checkpoint-last.pt), and any attached DATASET
# holding an uploaded checkpoint (Kaggle unpacks an uploaded .pt into a folder
# with data.pkl / data/ / byteorder / version -- it is zipped back here).
# A checkpoint is used only if its weights match this config's model exactly
# (same names and shapes) AND its path contains "diagonal". Shapes alone are not
# enough here: the row and prefix K=8 arms have the SAME state_dict and checkpoints
# do not record the layout, so their checkpoints would load silently. Keep
# "diagonal" in the name of any dataset you upload a checkpoint to.
# The newest epoch wins. To force one, set RESUME_CKPT_FORCE to its path.
RESUME_CKPT_FORCE = None
import os, glob, zipfile, torch, yaml
from scripts.evaluate import build_model
_ref = {k: tuple(v.shape) for k, v in build_model(yaml.safe_load(open(CONFIG))).state_dict().items()}

def _rebuild(folder):
    """Zip an unpacked checkpoint folder back into a loadable .pt in /tmp."""
    dst = os.path.join("/tmp/resume", folder.strip("/").replace("/", "__") + ".pt")
    if not os.path.exists(dst):
        os.makedirs("/tmp/resume", exist_ok=True)
        with zipfile.ZipFile(dst, "w", zipfile.ZIP_STORED) as z:
            for root, _, files in os.walk(folder):
                for f in files:
                    full = os.path.join(root, f)
                    z.write(full, os.path.join("checkpoint", os.path.relpath(full, folder)))
    return dst

cands = [os.path.join(OUT_DIR, "checkpoint-last.pt")]
cands += sorted(glob.glob("/kaggle/input/**/checkpoint-last.pt", recursive=True))
cands += [_rebuild(os.path.dirname(p))
          for p in sorted(glob.glob("/kaggle/input/**/data.pkl", recursive=True))]
best = None
print("checkpoints found:")
for c in ([RESUME_CKPT_FORCE] if RESUME_CKPT_FORCE else cands):
    if not (c and os.path.exists(c)):
        continue
    try:
        p = torch.load(c, map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"  unreadable ({type(e).__name__}): {c}")
        continue
    shapes = {k: tuple(v.shape) for k, v in p.get("model", {}).items()} == _ref
    match = shapes and (c == RESUME_CKPT_FORCE or "diagonal" in c.lower())
    label = "THIS model " if match else ("other arm? " if shapes else "other model")
    print(f"  {label}  epoch {p.get('epoch')}  {c}")
    if match and (best is None or p["epoch"] > best[0]):
        best = (p["epoch"], c)
    del p
RESUME_CKPT = best[1] if best else None

print("config :", CONFIG, "| layout: diagonal | content:", CONTENT, "| len:", LENGTH, "| blocks:", f"{START}-11")
print("data   :", DATA_DIR, "(exists:", os.path.isdir(DATA_DIR), ")")
print("out    :", OUT_DIR)
print("resume :", f"{RESUME_CKPT} (last finished epoch {best[0]})" if best else "(fresh start)")

## 6. Train  *(Save Version before 12h; next session it resumes by itself)*

In [ ]:
# ── TRAIN ──────────────────────────────────────────────────────────────
# data_dir + the model knobs now live in the (edited) config; we only override run-specific
# knobs here. Runs in a subprocess (fresh torch, no restart).
CMD = (f"python scripts/run_experiment.py --config {CONFIG} "
       f"checkpoint.output_dir={OUT_DIR} checkpoint.save_last_freq={SAVE_FREQ} "
       f"training.epochs={EPOCHS}")
if RESUME_CKPT:
    CMD += f" checkpoint.resume_from={RESUME_CKPT}"
print(CMD, "\n" + "="*70)
!{CMD}

## 7. Evaluate  *(comment out the whole cell to skip)*

In [ ]:
import glob, os

# Find the checkpoint to evaluate: this session's OUT_DIR first, then any
# attached notebook output (Save Version -> attach it as an input). Searches
# instead of hardcoding a notebook hash, so nothing needs editing after the
# first Save Version.
cands = []
for root in (os.path.dirname(OUT_DIR), "/kaggle/input"):
    for which in ("checkpoint-best.pt", "checkpoint-last.pt"):
        cands += sorted(glob.glob(f"{root}/**/{RUN}/{which}", recursive=True))
assert cands, f"no checkpoint found for {RUN} -- train first, or attach the saved Version"
EVAL_CKPT = cands[0]
EVAL_OUT  = f"/kaggle/working/{RUN}/eval"
print("eval ckpt:", EVAL_CKPT)

EVAL_CMD = (f"python scripts/evaluate.py --config {CONFIG} --checkpoint {EVAL_CKPT} "
            f"--n_samples 10000 --batch_size 200 --ema 1 "
            f"--cfg_scale 2.5 --cfg_interval 0.1 1.0 --out_dir {EVAL_OUT}")
print(EVAL_CMD, "\n" + "="*70)
!PYTHONPATH=. {EVAL_CMD}

## Notes
- Checkpoints (`-last`/`-best`/`-ep###`) with both EMA copies + optimizer are in `OUT_DIR` under `/kaggle/working`, so a resumed run equals a continuous one.
- **Resuming, also on another Kaggle account:** attach the previous session's notebook output, or upload its `checkpoint-last.pt` as a dataset **with `diagonal` in the dataset name** and attach that. The settings cell keeps only checkpoints with this model's shapes and `diagonal` in the path, and resumes from the newest epoch. A row or prefix K=8 checkpoint has the same shapes and shows as `other arm?`; to use a checkpoint regardless, set `RESUME_CKPT_FORCE`. Check the `resume :` line before training.
- **Data:** the Tiny-ImageNet cell uses an attached dataset first, then Stanford, then the SJTU mirror of the identical zip (MD5-checked). To skip downloads for good, upload `tiny-imagenet-200.zip` once as your own Kaggle dataset and attach it.
- Keep `EPOCHS`, `cfg_scale`, `DATA_DIR`, seed and sampler identical to the other runs, so the only variable is the placement.
- **Controls with the same 8 tokens:** `jit-s2-vmamba-incontext-row-class.yaml`, and `jit-s2-vmamba-incontext-class.yaml` with `model.in_context_len=8`. The existing prefix K-sweep (K=1/16/32 → 89.20 / 89.51 / 89.44) has no K=8 run.
- Added params are only `incontext_pos_embed` (1, 8, 384), identical to the row arm and prefix K=8. The extra cost is sequence length in the last 8 of 12 blocks plus one gather per path; record the it/s and peak memory printed by the GPU check for the complexity table.
- Sparser variant: `LENGTH = 4` puts tokens on (0,0) (2,2) (4,4) (6,6), one every 18 scan steps; every token still reads and writes the whole grid. Change `RUN` too, keeping `diagonal` in it, so the two runs' checkpoints stay apart.
